In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import urllib.request
import math

In [3]:
print("Downloading Shakespeare dataset...")
url = "https://ocw.mit.edu/ans7870/6/6.006/s08/lecturenotes/files/t8.shakespeare.txt"
response = urllib.request.urlopen(url)
text = response.read().decode('utf-8')

In [4]:
text = text[10000:60000]
chars = sorted(list(set(text)))
vocab_size = len(chars)

char_to_ix = {ch: i for i, ch in enumerate(chars)}
ix_to_char = {i: ch for i, ch in enumerate(chars)}

In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super(MultiHeadAttention, self).__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads

        assert (self.head_dim * heads == embed_size), "Embed size needs to be div by heads"

        self.values = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.keys = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.queries = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.fc_out = nn.Linear(heads * self.head_dim, embed_size)

    def forward(self, values, keys, query, mask):
        N = query.shape[0]
        value_len, key_len, query_len = values.shape[1], keys.shape[1], query.shape[1]

        values = values.reshape(N, value_len, self.heads, self.head_dim)
        keys = keys.reshape(N, key_len, self.heads, self.head_dim)
        queries = query.reshape(N, query_len, self.heads, self.head_dim)

        energy = torch.einsum("nqhd,nkhd->nhqk", [queries, keys])
        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e20"))

        attention = torch.softmax(energy / (self.embed_size ** (1/2)), dim=3)
        out = torch.einsum("nhql,nlhd->nqhd", [attention, values]).reshape(
            N, query_len, self.heads * self.head_dim
        )
        return self.fc_out(out)

class TransformerBlock(nn.Module):
    #encoder
    def __init__(self, embed_size, heads, dropout, forward_expansion):
        super(TransformerBlock, self).__init__()
        self.attention = MultiHeadAttention(embed_size, heads)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)

        self.feed_forward = nn.Sequential(
            nn.Linear(embed_size, forward_expansion * embed_size),
            nn.ReLU(),
            nn.Linear(forward_expansion * embed_size, embed_size)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, value, key, query, mask):
        attention = self.attention(value, key, query, mask)
        x = self.dropout(self.norm1(attention + query))
        forward = self.feed_forward(x)
        out = self.dropout(self.norm2(forward + x))
        return out

class DecoderBlock(nn.Module):
    def __init__(self, embed_size, heads, forward_expansion, dropout):
        super(DecoderBlock, self).__init__()
        self.attention = MultiHeadAttention(embed_size, heads)
        self.norm = nn.LayerNorm(embed_size)
        self.transformer_block = TransformerBlock(embed_size, heads, dropout, forward_expansion)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, value, key, src_mask, trg_mask):
        # Masked Self-Attention
        attention = self.attention(x, x, x, trg_mask)
        query = self.dropout(self.norm(attention + x))

        # Cross-Attention with Encoder outputs (value, key)
        out = self.transformer_block(value, key, query, src_mask)
        return out

In [6]:
class TransformerScratch(nn.Module):
    def __init__(self, vocab_size, embed_size=64, num_layers=2, heads=4, forward_expansion=2, dropout=0.1, max_length=100):
        super(TransformerScratch, self).__init__()
        # Embeddings & Positional Encoding
        self.word_embedding = nn.Embedding(vocab_size, embed_size)
        self.position_embedding = nn.Embedding(max_length, embed_size)

        # Encoder Stack
        self.encoder_layers = nn.ModuleList(
            [TransformerBlock(embed_size, heads, dropout, forward_expansion) for _ in range(num_layers)]
        )

        # Decoder Stack
        self.decoder_layers = nn.ModuleList(
            [DecoderBlock(embed_size, heads, forward_expansion, dropout) for _ in range(num_layers)]
        )

        self.fc_out = nn.Linear(embed_size, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def make_trg_mask(self, trg):
        N, trg_len = trg.shape
        # Lower triangular matrix for look-ahead mask
        trg_mask = torch.tril(torch.ones((trg_len, trg_len))).expand(
            N, 1, trg_len, trg_len
        ).to(trg.device)
        return trg_mask

    def forward(self, src, trg):
        N, seq_length = src.shape
        positions = torch.arange(0, seq_length).expand(N, seq_length).to(src.device)

        # 1. Encode
        src_embed = self.dropout(self.word_embedding(src) + self.position_embedding(positions))
        enc_out = src_embed
        for layer in self.encoder_layers:
            enc_out = layer(enc_out, enc_out, enc_out, mask=None)

        # 2. Decode
        trg_mask = self.make_trg_mask(trg)
        trg_embed = self.dropout(self.word_embedding(trg) + self.position_embedding(positions))

        dec_out = trg_embed
        for layer in self.decoder_layers:
            dec_out = layer(dec_out, enc_out, enc_out, src_mask=None, trg_mask=trg_mask)

        # 3. Output
        out = self.fc_out(dec_out)
        return out

In [7]:
seq_length = 32
batch_size = 16

def get_batch():
    # Randomly sample context windows
    ix = torch.randint(0, len(text) - seq_length - 1, (batch_size,))
    x = torch.stack([torch.tensor([char_to_ix[ch] for ch in text[i:i+seq_length]]) for i in ix])
    y = torch.stack([torch.tensor([char_to_ix[ch] for ch in text[i+1:i+seq_length+1]]) for i in ix])
    return x, y

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TransformerScratch(vocab_size=vocab_size, max_length=seq_length).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

print("Training model...")
model.train()
for epoch in range(100):
    src, trg = get_batch()
    src, trg = src.to(device), trg.to(device)

    # In seq2seq modeling of the same sequence, target input is shifted
    optimizer.zero_grad()
    output = model(src, src) # Using src as the shifted target for auto-regressive context

    # Flatten the output and targets for loss calculation
    output = output.reshape(-1, output.shape[2])
    trg = trg.reshape(-1)

    loss = criterion(output, trg)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

print("Training completed.")

Training model...
Epoch 0 | Loss: 4.4743
Epoch 20 | Loss: 3.1100
Epoch 40 | Loss: 3.0348
Epoch 60 | Loss: 2.8228
Epoch 80 | Loss: 2.8039
Training completed.
